# Train CSRNet on Colab

Train one ShanghaiTech part on a Colab GPU. Checkpoints save to Google Drive every epoch. After a disconnect, run all cells again. `--resume` continues from `last.pth`.

## Before you start

- [ ] Select **Runtime > Change runtime type > T4 GPU**.
- [ ] Upload these items to Drive under `MyDrive/crowd-estimator/`:
  - `ml/`
  - `ShanghaiTech/` (raw dataset, 210 MB)
  - `requirements-colab.txt`
- [ ] Skip `data/processed`. Step 4 rebuilds it in about one minute.

## 1. Check the GPU

Stop early on a CPU runtime. CPU training runs about 20 times slower.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Select Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Mount Drive and set parameters

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/crowd-estimator"  # holds ml/, ShanghaiTech/, requirements-colab.txt
PART = "part_A"  # "part_A" or "part_B"
EPOCHS = 150

## 3. Install dependencies

Keep the Colab CUDA build of `torch`. Install the other packages, then install `crowd` without its dependencies.

In [ ]:
!pip install -q -r {ROOT}/requirements-colab.txt
!pip install -q --no-deps -e {ROOT}/ml

import torch

assert torch.cuda.is_available(), "pip replaced the CUDA build of torch."


## 4. Build density maps

Write the maps to local disk, not Drive. The build skips files that exist.

In [ ]:
!python -m crowd.data.build --raw {ROOT}/ShanghaiTech --out /content/processed

## 5. Train

Start a new run, or resume the latest `PART` run. Outputs:

- Checkpoints and metrics: `{ROOT}/runs/<run_id>/`
- Run summary: `{ROOT}/docs/runs/<run_id>.json`

Stop training after 25 epochs without a validation MAE gain (`--patience 25`).

In [ ]:
!python -m crowd.train --part {PART} --epochs {EPOCHS} --resume --patience 25 \
    --raw {ROOT}/ShanghaiTech --processed /content/processed \
    --runs-dir {ROOT}/runs --publish-dir {ROOT}/docs/runs

## 6. Review the run

Plot validation error for the latest run and print its summary. Then:

- [ ] Copy `{ROOT}/docs/runs/<run_id>.json` to `docs/runs/` in the repo.
- [ ] Download `{ROOT}/runs/<run_id>/best.pth`.

In [ ]:
import json
import pathlib

import pandas as pd

runs = pathlib.Path(f"{ROOT}/runs").glob(f"{PART}-*")
run = max(runs, key=lambda p: p.stat().st_mtime)

with open(run / "metrics.jsonl") as f:
    metrics = pd.DataFrame([json.loads(line) for line in f]).set_index("epoch")
metrics[["val_mae", "val_rmse"]].dropna().plot(title=f"{run.name} validation error")

with open(run / "summary.json") as f:
    print(json.dumps(json.load(f), indent=2))